In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import OrderedDict
from Bio import SeqIO
from pysam import FastaFile
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))
from akita_model.model import SeqNN

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
GENOME_FASTA = "/project2/fudenber_735/genomes/mm10/mm10.fa"
BACKGROUND_FASTA = (
    "/project2/fudenber_735/smaruj/sequence_design/"
    "ledidi_semifreddo_akita/background_generation/"
    "background_sequences_scd30_totvar1300.fasta"
)
CTCF_TSV = (
    "/home1/smaruj/akitaV2-analyses/input_data/select_top20percent/"
    "output/CTCFs_jaspar_filtered_mm10_top20percent.tsv"
)
SINEB2_WITH_CTCF = "sineB2_with_ctcf_300.tsv"
SINEB2_NO_CTCF = "sineB2_no_ctcf_300.tsv"

MODEL_PATH = (
    "/home1/smaruj/pytorch_akita/models/finetuned/mouse/"
    "Hsieh2019_mESC/checkpoints/"
    "Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth"
)

TARGET_LEN = 220   # pad all sequences to this
CTCF_FLANK = 30
N_SAMPLES = 300
TOP_K = 20
OUTPUT_DIR = "filter_activation_analysis"
BATCH_SIZE = 64    # 220bp sequences are tiny, can use big batches

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL — adjust import
# ═══════════════════════════════════════════════════════════════════════════════

def load_model():
    model = SeqNN()
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    print(f"Model loaded on {DEVICE}")
    return model

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. SEQUENCE EXTRACTION & PADDING
# ═══════════════════════════════════════════════════════════════════════════════

def one_hot_encode(seq):
    mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
    enc = np.zeros((4, len(seq)), dtype=np.float32)
    for i, b in enumerate(seq.upper()):
        if b in mapping:
            enc[mapping[b], i] = 1.0
        else:
            enc[:, i] = 0.25
    return enc


def load_background_sequence(path):
    return str(next(SeqIO.parse(path, "fasta")).seq)


def pad_to_target(seq, target_len, bg_seq, bg_offset=0):
    if len(seq) >= target_len:
        excess = len(seq) - target_len
        s = excess // 2
        return seq[s:s + target_len], bg_offset
    pad_total = target_len - len(seq)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    bg_len = len(bg_seq)
    left = "".join(bg_seq[(bg_offset + i) % bg_len] for i in range(pad_left))
    bg_offset = (bg_offset + pad_left) % bg_len
    right = "".join(bg_seq[(bg_offset + i) % bg_len] for i in range(pad_right))
    bg_offset = (bg_offset + pad_right) % bg_len
    return left + seq + right, bg_offset


def extract_sequences(df, genome, bg_seq, chrom_col, start_col, end_col,
                      flank=0, target_len=TARGET_LEN, n_samples=N_SAMPLES):
    seqs = []
    bg_off = 0
    for _, row in df.head(n_samples).iterrows():
        chrom = str(row[chrom_col])
        start = max(0, int(row[start_col]) - flank)
        end = int(row[end_col]) + flank
        try:
            seq = genome.fetch(chrom, start, end)
        except (KeyError, ValueError):
            continue
        padded, bg_off = pad_to_target(seq, target_len, bg_seq, bg_off)
        assert len(padded) == target_len
        seqs.append(one_hot_encode(padded))
    return seqs


def prepare_all_groups():
    genome = FastaFile(GENOME_FASTA)
    bg = load_background_sequence(BACKGROUND_FASTA)

    # Group 1: CTCF
    ctcf_df = pd.read_csv(CTCF_TSV, sep="\t")
    print(f"CTCF df: {ctcf_df.shape}, cols: {ctcf_df.columns.tolist()}")
    def find_col(df, cands):
        for c in df.columns:
            if c.lower() in cands:
                return c
        raise ValueError(f"No col matching {cands}")

    cc = find_col(ctcf_df, {"chrom","chr","chromosome","sequence_name"})
    cs = find_col(ctcf_df, {"start","chromstart","genostart"})
    ce = find_col(ctcf_df, {"end","chromend","genoend"})
    g1 = extract_sequences(ctcf_df, genome, bg, cc, cs, ce,
                           flank=CTCF_FLANK, target_len=TARGET_LEN)
    print(f"Group 1 (CTCF ±{CTCF_FLANK}bp → {TARGET_LEN}bp): {len(g1)}")

    # Group 2: B2 + CTCF
    df2 = pd.read_csv(SINEB2_WITH_CTCF, sep="\t")
    g2 = extract_sequences(df2, genome, bg, "genoName","genoStart","genoEnd",
                           flank=0, target_len=TARGET_LEN)
    print(f"Group 2 (B2+CTCF → {TARGET_LEN}bp): {len(g2)}")

    # Group 3: B2 no CTCF
    df3 = pd.read_csv(SINEB2_NO_CTCF, sep="\t")
    g3 = extract_sequences(df3, genome, bg, "genoName","genoStart","genoEnd",
                           flank=0, target_len=TARGET_LEN)
    print(f"Group 3 (B2 noCTCF → {TARGET_LEN}bp): {len(g3)}")

    genome.close()
    return g1, g2, g3

In [5]:
def get_trunk_layers(model):
    """
    Extract early trunk layers, splitting conv_tower into individual
    blocks of 4 ops each: [ReLU, Conv1d, BN, MaxPool].
    
    conv_block_1: 220 → 110  (pool2)
    conv_tower block 0: 110 → 55
    conv_tower block 1:  55 → 27
    conv_tower block 2:  27 → 13
    conv_tower block 3:  13 →  6
    conv_tower block 4:   6 →  3
    conv_tower block 5:   3 →  1  ← last usable
    """
    layers = OrderedDict()
    
    # First conv block
    layers["conv_block_1"] = model.conv_block_1
    
    # Split conv_tower.conv_tower Sequential into blocks of 4
    tower_seq = model.conv_tower.conv_tower  # the inner Sequential
    ops = list(tower_seq.children())
    
    for i in range(0, len(ops), 4):
        block = nn.Sequential(*ops[i:i+4])
        layers[f"conv_tower_block{i//4}"] = block
    
    # Residual 1D blocks (no pooling — spatial dim preserved)
    for i in range(1, 12):
        name = f"residual1d_block{i}"
        if hasattr(model, name):
            layers[name] = getattr(model, name)
    
    # Channel reduction
    layers["conv_reduce"] = model.conv_reduce
    
    return layers

In [6]:
g1, g2, g3 = prepare_all_groups()

CTCF df: (300, 6), cols: ['chrom', 'end', 'start', 'strand', 'disruption_SCD', 'insertion_SCD']
Group 1 (CTCF ±30bp → 220bp): 300
Group 2 (B2+CTCF → 220bp): 300
Group 3 (B2 noCTCF → 220bp): 300


In [7]:
model = load_model()
trunk_layers = get_trunk_layers(model)

Model loaded on cuda:0


In [8]:
from filter_combo_analysis import run_combo_analysis

In [9]:
run_combo_analysis(model, g1, g2, g3, trunk_layers)

PART 1 — First-layer co-activation

Co-activation between known filters:
  B2_conserved(36) × BoxB(122):  CTCF=0.126  B2+CTCF=0.614  B2no=0.438
  B2_conserved(36) × B2_CTCF(104):  CTCF=0.054  B2+CTCF=0.333  B2no=0.083
  B2_conserved(36) × filter21(21):  CTCF=0.017  B2+CTCF=0.392  B2no=0.261
  BoxB(122) × B2_CTCF(104):  CTCF=0.035  B2+CTCF=0.236  B2no=0.048
  BoxB(122) × filter21(21):  CTCF=-0.064  B2+CTCF=0.424  B2no=0.117
  B2_CTCF(104) × filter21(21):  CTCF=-0.011  B2+CTCF=0.343  B2no=-0.016

Pairs more co-activated in B2+CTCF than CTCF:
  filters  15- 45  Δ=0.648  (CTCF=-0.102, B2+CTCF=0.546, B2no=0.238)
  filters   7- 15  Δ=0.641  (CTCF=-0.089, B2+CTCF=0.552, B2no=0.265)
  filters  14- 15  Δ=0.639  (CTCF=-0.042, B2+CTCF=0.598, B2no=0.261)
  filters  32-106  Δ=0.632  (CTCF=-0.123, B2+CTCF=0.509, B2no=0.136)
  filters  14- 45  Δ=0.623  (CTCF=-0.069, B2+CTCF=0.554, B2no=0.353)
  filters  45- 69  Δ=0.605  (CTCF=-0.155, B2+CTCF=0.450, B2no=0.387)
  filters  11- 15  Δ=0.602  (CTCF=-0.131

In [10]:
N_TOP = 50
FILTERS_OF_INTEREST = [15, 103]

INDEX_TO_BASE = {0: "A", 1: "C", 2: "G", 3: "T"}

In [11]:
def get_first_conv(model):
    for module in model.conv_block_1.modules():
        if isinstance(module, nn.Conv1d):
            return module
    raise ValueError("No Conv1d in conv_block_1")


def scan_filter(sequences, conv, filt_idx, device):
    """
    Scan all sequences with a single filter, return per-position activations.
    Returns sorted list of (seq_idx, pos, activation, subsequence_string).
    """
    w = conv.weight[filt_idx:filt_idx + 1].to(device)
    b = conv.bias[filt_idx:filt_idx + 1].to(device) if conv.bias is not None else None
    k = w.shape[2]

    results = []
    for si, oh in enumerate(sequences):
        t = torch.tensor(oh, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            act = torch.nn.functional.conv1d(t, w, bias=b).squeeze().cpu().numpy()
        for pos in range(len(act)):
            subseq = "".join(INDEX_TO_BASE[oh[:, pos + j].argmax()] for j in range(k))
            results.append((si, pos, act[pos], subseq))

    results.sort(key=lambda x: x[2], reverse=True)
    return results

def build_pwm(subseqs, k=15):
    """Build PWM from list of subsequence strings."""
    pfm = np.zeros((4, k))
    base_map = {"A": 0, "C": 1, "G": 2, "T": 3}
    for s in subseqs:
        for j, b in enumerate(s[:k]):
            if b in base_map:
                pfm[base_map[b], j] += 1
    pwm = pfm / pfm.sum(axis=0, keepdims=True)
    return pwm


def consensus_from_pwm(pwm):
    bases = "ACGT"
    return "".join(bases[i] for i in pwm.argmax(axis=0))

def extract_top_subseqs(model, g1, g2, g3, device):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    conv = get_first_conv(model)
    k = conv.weight.shape[2]
    print(f"First conv: {conv.weight.shape}, kernel={k}")

    groups = {"CTCF": g1, "B2+CTCF": g2, "B2_noCTCF": g3}

    for filt_idx in FILTERS_OF_INTEREST:
        print(f"\n{'='*65}")
        print(f"FILTER {filt_idx}")
        print(f"{'='*65}")

        for gname, seqs in groups.items():
            results = scan_filter(seqs, conv, filt_idx, device)
            top = results[:N_TOP]

            # Build consensus from top subsequences
            top_seqs = [r[3] for r in top]
            pwm = build_pwm(top_seqs, k)
            cons = consensus_from_pwm(pwm)

            mean_act = np.mean([r[2] for r in top])
            max_act = top[0][2]

            print(f"\n  {gname}:")
            print(f"    Consensus:   {cons}")
            print(f"    Mean act:    {mean_act:.3f}")
            print(f"    Max act:     {max_act:.3f}")
            print(f"    Top 10 subsequences:")
            for si, pos, act, seq in top[:10]:
                print(f"      seq={si:3d}  pos={pos:3d}  act={act:.3f}  {seq}")

            # Save all top subsequences
            with open(f"{OUTPUT_DIR}/filter{filt_idx}_{gname}_top{N_TOP}.txt", "w") as f:
                f.write(f"# Filter {filt_idx} — {gname}\n")
                f.write(f"# Consensus: {cons}\n")
                f.write(f"# Mean activation: {mean_act:.3f}\n")
                f.write("seq_idx\tposition\tactivation\tsubsequence\n")
                for si, pos, act, seq in top:
                    f.write(f"{si}\t{pos}\t{act:.4f}\t{seq}\n")

        # Compare consensus across groups
        print(f"\n  Cross-group comparison for filter {filt_idx}:")
        for gname, seqs in groups.items():
            results = scan_filter(seqs, conv, filt_idx, device)
            # Overall mean activation (all positions, all sequences)
            all_acts = [r[2] for r in results]
            mean_all = np.mean(all_acts)
            # Fraction of positions with activation > 0
            frac_pos = np.mean([a > 0 for a in all_acts])
            print(f"    {gname:12s}: mean_all={mean_all:.3f}  frac_positive={frac_pos:.3f}")


In [12]:
extract_top_subseqs(model, g1, g2, g3, DEVICE)

First conv: torch.Size([128, 4, 15]), kernel=15

FILTER 15

  CTCF:
    Consensus:   CACCAGGGGGCGGCA
    Mean act:    1.160
    Max act:     1.362
    Top 10 subsequences:
      seq=266  pos=107  act=1.362  CTCTAGTGGAGGAAA
      seq=213  pos=101  act=1.298  CTCCACTAGAGGGCA
      seq= 12  pos=104  act=1.292  CACCAGGTGGCGCTA
      seq=257  pos= 16  act=1.270  CTCTAGGGGAGGGCA
      seq=151  pos=104  act=1.254  CTCCAGGTGGCGCTG
      seq= 68  pos=104  act=1.252  CACTAGGTGGCGACA
      seq=271  pos=101  act=1.251  CGCCACTAGAGGGCA
      seq=170  pos=104  act=1.227  CACTAGGTGGCGCCA
      seq=295  pos=107  act=1.227  CTCTTGTGGCAGAAA
      seq= 40  pos=104  act=1.220  CACCAGAGGGCGAGA

  B2+CTCF:
    Consensus:   CACCAGAAGAGGGCA
    Mean act:    1.241
    Max act:     1.363
    Top 10 subsequences:
      seq=272  pos= 76  act=1.363  CACCAGAAGGCGGCA
      seq=174  pos= 79  act=1.358  CTCCAGAAGAAGGCA
      seq= 22  pos= 84  act=1.347  CACCAGAAGACGGCA
      seq=157  pos= 75  act=1.326  CTCCAGAAGAGGGC

In [13]:
from detect_b2_filters_layer4 import run_b2_filter_tracking

In [14]:
df = pd.read_csv(f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/only_successful_seqs_with_results.tsv", sep="\t")

In [ ]:
df.columns

In [15]:
run_b2_filter_tracking(
    model, g1, g2, g3, trunk_layers,
    coord_df=df,
    init_seq_path="/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X",
    slice_path="/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated",
)

Target layer: conv_tower_block3 (idx=4)

Step 1 — 220bp reference group activations
  Computing activations for CTCF group...
    Shape: (300, 128, 6)
  Computing activations for B2+CTCF group...
    Shape: (300, 128, 6)
  Computing activations for B2 noCTCF group...
    Shape: (300, 128, 6)

Step 2 — Identify B2-detecting filters

  Top 10 filters by B2+CTCF activation:
 filter_idx      CTCF   B2CTCF  B2noCTCF  B2CTCF_specificity
         87 11.339191 3.463077  1.226126           -2.819582
        127  8.469207 3.453486  1.470233           -1.516234
         51  6.950085 3.327150  1.594034           -0.944909
         15  7.132625 3.293620  1.520599           -1.032992
         68  2.458230 3.276452  1.797147            1.148763
         56  3.552344 2.997103  1.761981            0.339941
         13  6.814229 2.993022  1.360364           -1.094274
        108  8.429913 2.930835  1.296421           -1.932332
        117  3.506089 2.831225  1.714483            0.220939
         90  9.1

(     filter_idx      CTCF    B2CTCF  B2noCTCF  B2CTCF_vs_CTCF  B2CTCF_vs_B2no  \
 0             0  1.897376  1.272617  1.422636       -0.624759       -0.150019   
 1             1  0.758234  1.066796  1.015419        0.308562        0.051376   
 2             2  2.338444  0.495186  0.384467       -1.843258        0.110719   
 3             3  4.287790  1.349661  0.859518       -2.938129        0.490143   
 4             4  1.501282  0.935057  1.163984       -0.566225       -0.228927   
 ..          ...       ...       ...       ...             ...             ...   
 123         123  2.266597  0.909957  1.173386       -1.356640       -0.263429   
 124         124  1.033630  1.244460  1.227633        0.210830        0.016827   
 125         125  4.641395  1.755937  1.002678       -2.885458        0.753258   
 126         126  2.017107  0.852679  0.786100       -1.164429        0.066579   
 127         127  8.469207  3.453486  1.470233       -5.015721        1.983253   
 
      B2CTCF_s

In [16]:
from plot_boxplots import plot_separate_boxplots

In [17]:
from detect_b2_filters_layer4 import partial_forward_to_layer, partial_forward_tensors, load_init_bins, load_slices

# 220bp reference groups (already in memory as g1, g2, g3)
acts_ctcf = partial_forward_to_layer(g1, trunk_layers, 4, batch_size=64)
acts_b2ctcf = partial_forward_to_layer(g2, trunk_layers, 4, batch_size=64)
acts_b2no = partial_forward_to_layer(g3, trunk_layers, 4, batch_size=64)

In [18]:
# 2048bp before/after (if not still loaded)
before_tensors, _ = load_init_bins(df, "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X")
after_tensors, _ = load_slices(df, "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated")
acts_before = partial_forward_tensors(before_tensors, trunk_layers, 4, batch_size=16)
acts_after = partial_forward_tensors(after_tensors, trunk_layers, 4, batch_size=16)

  Loaded 348 init bins (of 348 requested)
  Shape: torch.Size([4, 2048])
  Loaded 348 slices (of 348 requested)
  Shape: torch.Size([4, 2048])


In [19]:
plot_separate_boxplots(acts_ctcf, acts_b2ctcf, acts_b2no,
                       acts_before, acts_after,
                       top_filters=[68, 81, 22])

Saved: b2_filter_tracking/boxplot_before_after_top3.svg
Saved: b2_filter_tracking/boxplot_references_top3.svg


[68, 81, 22]

In [20]:
from plot_filter_volcano import plot_filter_volcano

In [21]:
deltas, pvals, neg_log_p = plot_filter_volcano(
    acts_before, acts_after,
    highlight_filters=[68, 81, 22],
    highlight_labels={68: "Filter 68\n(B2+CTCF top1)", 
                      81: "Filter 81\n(B2+CTCF top2)", 
                      22: "Filter 22\n(B2+CTCF top3)"}
)

Saved: b2_filter_tracking/volcano_filters.svg

Highlighted filters:
  Filter 68: Δ=+7.71, p=8.68e-59, -log10(p)=58.1
  Filter 81: Δ=+2.80, p=1.22e-58, -log10(p)=57.9
  Filter 22: Δ=-0.42, p=5.15e-14, -log10(p)=13.3

Top 10 filters by |Δ activation|:
  Filter  28: Δ=-8.05, p=1.78e-58
  Filter  68: Δ=+7.71, p=8.68e-59 ★
  Filter  56: Δ=+7.46, p=1.39e-58
  Filter   7: Δ=-6.26, p=1.27e-58
  Filter  94: Δ=-5.83, p=2.24e-58
  Filter 117: Δ=+5.74, p=3.72e-58
  Filter  93: Δ=-5.71, p=8.61e-59
  Filter   9: Δ=+5.65, p=1.27e-58
  Filter  17: Δ=-5.63, p=1.42e-58
  Filter  71: Δ=-5.58, p=1.15e-58

Rank of highlighted filters (by Δ, descending):
  Filter 68: rank 1/128
  Filter 81: rank 6/128
  Filter 22: rank 45/128


In [ ]:
import ast
import re

In [ ]:
def get_first_conv(model):
    for module in model.conv_block_1.modules():
        if isinstance(module, nn.Conv1d):
            return module
    raise ValueError("No Conv1d in conv_block_1")


def parse_coord_set(coord_str):
    """
    Parse coordinate strings like:
      "{(586, 605, '+'), (1601, 1620, '+'), (6, 25, '-')}"
      or "set()"
    Returns list of (start, end, strand) tuples.
    """
    if pd.isna(coord_str) or coord_str.strip() in ("set()", "{}"):
        return []

    # Extract all (number, number, 'strand') patterns
    pattern = r"\((\d+),\s*(\d+),\s*'([+-])'\)"
    matches = re.findall(pattern, str(coord_str))
    return [(int(m[0]), int(m[1]), m[2]) for m in matches]

In [ ]:
def onehot_to_seq(onehot):
    """Convert one-hot tensor (4, L) to DNA string."""
    indices = onehot.argmax(dim=0)
    return "".join(INDEX_TO_BASE[i.item()] for i in indices)

In [ ]:
def scan_filter_at_positions(slice_tensor, conv, filter_idx, positions,
                              device=DEVICE):
    """
    Compute filter activation at specific positions in a sequence.

    Args:
        slice_tensor: (4, L) one-hot tensor
        conv: Conv1d layer
        filter_idx: which filter to use
        positions: list of (start, end, strand) tuples

    Returns:
        list of dicts with position info and activation values
    """
    w = conv.weight[filter_idx:filter_idx + 1].to(device)
    b = conv.bias[filter_idx:filter_idx + 1].to(device) if conv.bias is not None else None
    k = w.shape[2]  # kernel size (15)

    if slice_tensor.dim() == 3:
        slice_tensor = slice_tensor.squeeze(0)

    t = slice_tensor.unsqueeze(0).float().to(device)  # (1, 4, L)
    L = t.shape[2]

    # Run full convolution
    with torch.no_grad():
        full_acts = nn.functional.conv1d(t, w, bias=b).squeeze().cpu().numpy()  # (L - k + 1,)

    results = []
    for start, end, strand in positions:
        # The CTCF motif spans [start, end).
        # Find the conv position(s) that overlap with this motif.
        # Conv position p covers input [p, p+k).
        # We want the max activation in positions overlapping the motif.
        conv_start = max(0, start - k + 1)
        conv_end = min(len(full_acts), end)

        if conv_start >= conv_end or conv_start >= len(full_acts):
            continue

        region_acts = full_acts[conv_start:conv_end]
        max_act = region_acts.max()
        max_pos = conv_start + region_acts.argmax()
        mean_act = region_acts.mean()

        # Extract the subsequence at max activation position
        seq_start = max_pos
        seq_end = min(seq_start + k, L)
        if seq_end - seq_start < k:
            continue
        subseq = onehot_to_seq(slice_tensor[:, seq_start:seq_end])

        results.append({
            "ctcf_start": start,
            "ctcf_end": end,
            "strand": strand,
            "max_activation": max_act,
            "mean_activation": mean_act,
            "best_pos": max_pos,
            "best_subseq": subseq,
        })

    return results

In [ ]:
def scan_ctcfs_with_filter104(model, coord_df, slice_path,
                                filter_idx=104, has_fold=True,
                                save_dir=OUTPUT_DIR):
    """
    Main analysis: scan all original and new CTCFs with filter 104.

    Args:
        model: loaded AkitaV2
        coord_df: DataFrame with orig_CTCFs_coord, new_CTCFs_coord columns
        slice_path: path to optimized slice .pt files
        filter_idx: filter to scan with (default 104)
        has_fold: whether paths include fold subdirectory
    """
    os.makedirs(save_dir, exist_ok=True)
    conv = get_first_conv(model)
    k = conv.weight.shape[2]
    print(f"Scanning with filter {filter_idx} (kernel size {k})")

    all_orig = []
    all_new = []

    for idx, row in coord_df.iterrows():
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]

        # Load optimized slice
        if has_fold and "fold" in row.index:
            fold = row["fold"]
            fpath = f"{slice_path}/fold{fold}/{chrom}_{start}_{end}_slice.pt"
        else:
            fpath = f"{slice_path}/{chrom}_{start}_{end}_slice.pt"

        try:
            st = torch.load(fpath, weights_only=True, map_location="cpu")
        except FileNotFoundError:
            continue

        # Parse CTCF coordinates
        orig_ctcfs = parse_coord_set(row.get("orig_CTCFs_coord", "set()"))
        new_ctcfs = parse_coord_set(row.get("new_CTCFs_coord", "set()"))

        # Scan original CTCFs
        if orig_ctcfs:
            results = scan_filter_at_positions(st, conv, filter_idx, orig_ctcfs)
            for r in results:
                r["category"] = "pre-existing"
                r["chrom"] = chrom
                r["seq_start"] = start
                r["seq_end"] = end
                all_orig.append(r)

        # Scan new CTCFs
        if new_ctcfs:
            results = scan_filter_at_positions(st, conv, filter_idx, new_ctcfs)
            for r in results:
                r["category"] = "new"
                r["chrom"] = chrom
                r["seq_start"] = start
                r["seq_end"] = end
                all_new.append(r)

    df_orig = pd.DataFrame(all_orig)
    df_new = pd.DataFrame(all_new)
    df_all = pd.concat([df_orig, df_new], ignore_index=True)

    print(f"\nPre-existing CTCFs scanned: {len(df_orig)}")
    print(f"New CTCFs scanned:          {len(df_new)}")

    if len(df_orig):
        print(f"\nPre-existing CTCF filter {filter_idx} activations:")
        print(f"  Mean: {df_orig['max_activation'].mean():.3f}")
        print(f"  Median: {df_orig['max_activation'].median():.3f}")
        print(f"  Std: {df_orig['max_activation'].std():.3f}")

    if len(df_new):
        print(f"\nNew CTCF filter {filter_idx} activations:")
        print(f"  Mean: {df_new['max_activation'].mean():.3f}")
        print(f"  Median: {df_new['max_activation'].median():.3f}")
        print(f"  Std: {df_new['max_activation'].std():.3f}")

    # Save
    df_all.to_csv(f"{save_dir}/ctcf_filter{filter_idx}_scan.csv", index=False)

    # Pre-existing CTCFs are untouched by optimization (edits blocked there),
    # so we only compare pre-existing vs new in the optimized slice.
    
    # Plot 1: Boxplot comparing categories
    fig, ax = plt.subplots(figsize=(5, 5))

    plot_data = []
    plot_labels = []
    plot_colors = []

    if len(df_orig):
        plot_data.append(df_orig["max_activation"].values)
        plot_labels.append("Pre-existing\nCTCFs")
        plot_colors.append("#457B9D")

    if len(df_new):
        plot_data.append(df_new["max_activation"].values)
        plot_labels.append("New\nCTCFs")
        plot_colors.append("#E76F51")

    bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                    widths=0.55, showfliers=False)
    for patch, color in zip(bp["boxes"], plot_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
        patch.set_edgecolor("k")
        patch.set_linewidth(1)
    for median in bp["medians"]:
        median.set_color("k")
        median.set_linewidth(1.5)

    for j, (d, color) in enumerate(zip(plot_data, plot_colors)):
        jitter = np.random.normal(0, 0.05, len(d))
        ax.scatter(np.ones(len(d)) * (j + 1) + jitter, d,
                  s=10, alpha=0.3, c=color, edgecolors="none", zorder=3)

    ax.set_ylabel(f"Filter {filter_idx} max activation", fontsize=11)
    ax.set_title(f"Filter {filter_idx} (SINE B2 CTCF variant) at CTCF sites\n"
                 f"Are pre-existing CTCFs converted to B2-like?",
                 fontsize=11, fontweight="bold")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    fig.savefig(f"{save_dir}/ctcf_filter{filter_idx}_boxplot.png",
                dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"\nSaved: {save_dir}/ctcf_filter{filter_idx}_boxplot.png")

    # Plot 2: Histogram overlay
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bins = np.linspace(
        min(d.min() for d in plot_data),
        max(d.max() for d in plot_data),
        40
    )
    for d, label, color in zip(plot_data, plot_labels, plot_colors):
        ax.hist(d, bins=bins, alpha=0.5, label=label.replace("\n", " "),
                color=color, edgecolor="k", linewidth=0.3)

    ax.set_xlabel(f"Filter {filter_idx} max activation", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title(f"Distribution of filter {filter_idx} activation at CTCF sites",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    fig.savefig(f"{save_dir}/ctcf_filter{filter_idx}_histogram.png",
                dpi=200, bbox_inches="tight")
    plt.close(fig)
    # print(f"Saved: {save_dir}/ctcf_filter{filter_idx}_histogram.png")
    plt.show()

    # ── Summary statistics ───────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"SUMMARY — Filter {filter_idx} at CTCF sites")
    print(f"{'='*60}")

    if len(df_new) and len(df_orig):
        diff = df_new["max_activation"].mean() - df_orig["max_activation"].mean()
        print(f"\nNew vs pre-existing CTCFs: Δ = {diff:+.3f}")
        if diff > 0.5:
            print(f"  → New CTCFs are MORE B2-like than pre-existing ones")
            print(f"  → The B2-like CTCFs are predominantly NEW, not converted")
        elif diff < -0.5:
            print(f"  → New CTCFs are LESS B2-like than pre-existing ones")
        else:
            print(f"  → Similar B2-likeness between new and pre-existing")

    return df_all

In [ ]:
df_all = scan_ctcfs_with_filter104(
    model, df,
    slice_path="/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated",
    filter_idx=104,
)

In [ ]:
# Plot 1: Boxplot comparing categories
    fig, ax = plt.subplots(figsize=(5, 5))

    plot_data = []
    plot_labels = []
    plot_colors = []

    if len(df_orig):
        plot_data.append(df_orig["max_activation"].values)
        plot_labels.append("Pre-existing\nCTCFs")
        plot_colors.append("#457B9D")

    if len(df_new):
        plot_data.append(df_new["max_activation"].values)
        plot_labels.append("New\nCTCFs")
        plot_colors.append("#E76F51")

    bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                    widths=0.55, showfliers=False)
    for patch, color in zip(bp["boxes"], plot_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
        patch.set_edgecolor("k")
        patch.set_linewidth(1)
    for median in bp["medians"]:
        median.set_color("k")
        median.set_linewidth(1.5)

    for j, (d, color) in enumerate(zip(plot_data, plot_colors)):
        jitter = np.random.normal(0, 0.05, len(d))
        ax.scatter(np.ones(len(d)) * (j + 1) + jitter, d,
                  s=10, alpha=0.3, c=color, edgecolors="none", zorder=3)

    ax.set_ylabel(f"Filter {filter_idx} max activation", fontsize=11)
    ax.set_title(f"Filter {filter_idx} (SINE B2 CTCF variant) at CTCF sites\n"
                 f"Are pre-existing CTCFs converted to B2-like?",
                 fontsize=11, fontweight="bold")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    fig.savefig(f"{save_dir}/ctcf_filter{filter_idx}_boxplot.png",
                dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"\nSaved: {save_dir}/ctcf_filter{filter_idx}_boxplot.png")

    # Plot 2: Histogram overlay
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bins = np.linspace(
        min(d.min() for d in plot_data),
        max(d.max() for d in plot_data),
        40
    )
    for d, label, color in zip(plot_data, plot_labels, plot_colors):
        ax.hist(d, bins=bins, alpha=0.5, label=label.replace("\n", " "),
                color=color, edgecolor="k", linewidth=0.3)

    ax.set_xlabel(f"Filter {filter_idx} max activation", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title(f"Distribution of filter {filter_idx} activation at CTCF sites",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    fig.savefig(f"{save_dir}/ctcf_filter{filter_idx}_histogram.png",
                dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_dir}/ctcf_filter{filter_idx}_histogram.png")

    # ── Summary statistics ───────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"SUMMARY — Filter {filter_idx} at CTCF sites")
    print(f"{'='*60}")

    if len(df_new) and len(df_orig):
        diff = df_new["max_activation"].mean() - df_orig["max_activation"].mean()
        print(f"\nNew vs pre-existing CTCFs: Δ = {diff:+.3f}")
        if diff > 0.5:
            print(f"  → New CTCFs are MORE B2-like than pre-existing ones")
            print(f"  → The B2-like CTCFs are predominantly NEW, not converted")
        elif diff < -0.5:
            print(f"  → New CTCFs are LESS B2-like than pre-existing ones")
        else:
            print(f"  → Similar B2-likeness between new and pre-existing")

    return df_all

In [ ]:
# 2. Create the Reverse Complement Weights
# Flip the length (dim 2) and the channel order (dim 1: A/C/G/T -> T/G/C/A)
weights_104_rc = weights_104.flip(2).index_select(1, torch.tensor([3, 2, 1, 0]).to(weights_104.device))

In [ ]:
def get_manual_f104_score_nobias(tensor_4_2048, weights):
    """
    Performs manual 1D convolution without bias at 1bp resolution.
    """
    # Ensure tensor is [1, 4, 2048]
    x = tensor_4_2048.unsqueeze(0).to(weights.device)
    
    # Manual convolution with bias=None
    # padding=7 maintains the 2048bp length for a kernel of 15
    activation = F.conv1d(x, weights, bias=None, padding=7)
    
    # Remove batch and channel dims -> Shape (2048,)
    return activation.squeeze().detach().cpu().numpy()

In [ ]:
import ast

def parse_coord(val, max_len=2048):
    """
    Parses a string representation of a set of coordinate tuples.
    Format expected: {(start, end, strand), ...}
    Filters out coordinates that are negative or exceed max_len.
    """
    # 1. Handle cases where data might already be a set/list
    if not isinstance(val, str):
        data = val
    else:
        try:
            # 2. Safely evaluate the string into a Python set
            data = ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []

    # 3. Validation and Boundary Filtering
    valid_coords = []
    if data and isinstance(data, (set, list)):
        for coord in data:
            # Ensure it's a tuple with at least start and end
            if isinstance(coord, tuple) and len(coord) >= 2:
                start, end = coord[0], coord[1]
                
                # Boundary Check: 0 <= start < end <= 2048
                if 0 <= start < end <= max_len:
                    valid_coords.append(coord)
                    
    return valid_coords

In [ ]:
def get_dual_strand_score(tensor_4_2048, w_fwd, w_rc):
    x = tensor_4_2048.unsqueeze(0).to(w_fwd.device)
    
    # Convolution for both orientations
    act_fwd = F.conv1d(x, w_fwd, bias=None, padding=7).squeeze()
    act_rc = F.conv1d(x, w_rc, bias=None, padding=7).squeeze()
    
    # Take the element-wise maximum across strands
    # Shape remains (2048,)
    return torch.max(act_fwd, act_rc).detach().cpu().numpy()

results = []

In [ ]:
results = []

# Assuming your loop structure from before
for i in range(len(before_tensors)):
    # Calculate profiles
    act_init = get_manual_f104_score_nobias(before_tensors[i], weights_104)
    act_edit = get_manual_f104_score_nobias(after_tensors[i], weights_104)
    
    row = df.iloc[i]
    orig_coords = parse_coord(row['orig_CTCFs_coord'])
    new_coords = parse_coord(row['new_CTCFs_coord'])
    
    # Now [start:end] matches exactly at 1bp
    for (start, end, _) in orig_coords:
        results.append({
            'Type': 'Pre-existing', 
            'State': 'Initial', 
            'Score': np.max(act_init[start:end])
        })
        results.append({
            'Type': 'Pre-existing', 
            'State': 'Edited', 
            'Score': np.max(act_edit[start:end])
        })

    for (start, end, _) in new_coords:
        results.append({
            'Type': 'New', 
            'State': 'Edited', 
            'Score': np.max(act_edit[start:end])
        })

analysis_df = pd.DataFrame(results)

In [ ]:
results = []

for i in range(len(after_tensors)):
    # Get the "best of both strands" profile
    act_best = get_dual_strand_score(after_tensors[i], weights_104, weights_104_rc)
    
    row = df.iloc[i]
    orig_coords = parse_coord(row['orig_CTCFs_coord'])
    new_coords = parse_coord(row['new_CTCFs_coord'])
    
    for (start, end, _) in orig_coords:
        results.append({
            'CTCF Type': 'Pre-existing (Edited)', 
            'Filter 104 Score': np.max(act_best[start:end])
        })
            
    for (start, end, _) in new_coords:
        results.append({
            'CTCF Type': 'New (Edited)', 
            'Filter 104 Score': np.max(act_best[start:end])
        })

comparison_df = pd.DataFrame(results)

In [ ]:
comparison_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Set the preferred order of the groups
plot_order = ['Pre-existing (Edited)', 'New (Edited)']

# 3. Set up the visualization style
sns.set_style("whitegrid")
plt.figure(figsize=(10, 7))

# Define colors: Grey for baseline, Blue for modified, Red for synthetic
palette = {
    'Pre-existing (Edited)': '#3498db', 
    'New (Edited)': '#e74c3c'
}

# 4. Create the Boxplot
sns.boxplot(
    x='CTCF Type', 
    y='Filter 104 Score', 
    data=comparison_df, 
    order=plot_order, 
    palette=palette, 
    showfliers=False,
    width=0.6
)

# 5. Overlay individual points (Stripplot) to see the distribution density
# sns.stripplot(
#     x='Group', 
#     y='Score', 
#     data=analysis_df, 
#     order=plot_order, 
#     color='black', 
#     alpha=0.25, 
#     size=3,
#     jitter=0.2
# )

# 6. Formatting and Titles
plt.title('Modification vs. De Novo Creation of SINE B2-like CTCFs', fontsize=15, pad=20)
plt.ylabel('Filter 104 Activation Score\n(Manual Convolution @ 1bp resolution)', fontsize=12)
plt.xlabel('')

# Add a dashed line at the average Initial score for easier comparison
# baseline_mean = analysis_df[analysis_df['Group'] == 'Pre-existing (Initial)']['Score'].mean()
# plt.axhline(baseline_mean, color='black', linestyle='--', alpha=0.5, label='Baseline Mean')

plt.tight_layout()
# plt.savefig('ctcf_score_comparison.png', dpi=300)
plt.show()